In [1]:
import pandas as pd
import earthaccess as ea
from db_conn import get_folha_geom_geojson


In [2]:
def search_aster_l1t_004_cloudfree(
    bbox,
    start,
    end,
    cloud_max=5.0,
):
    temporal = (start, end)

    results = ea.search_data(
        short_name="AST_L1T",
        version="004",
        temporal=temporal,
        bounding_box=bbox,
        cloud_cover=(0, cloud_max) if cloud_max is not None else None,
        cloud_hosted=True,
        count=-1,
    )

    if not results:
        return [], pd.DataFrame()

    records = []
    for g in results:
        u = g["umm"]

        granule_id = u.get("GranuleUR")

        temporal_umm = u.get("TemporalExtent") or u.get("TemporalExtents")
        start_str = None

        if isinstance(temporal_umm, dict):
            rd = temporal_umm.get("RangeDateTime", {})
            if isinstance(rd, dict):
                start_str = rd.get("BeginningDateTime")
            if start_str is None:
                start_str = temporal_umm.get("SingleDateTime")
        elif isinstance(temporal_umm, list) and temporal_umm:
            te0 = temporal_umm[0]
            rd = te0.get("RangeDateTime", {})
            if isinstance(rd, dict):
                start_str = rd.get("BeginningDateTime")
            if start_str is None:
                start_str = te0.get("SingleDateTime")

        cc = None
        if "CloudCover" in u:
            cc = u["CloudCover"]
        else:
            dg = u.get("DataGranule", {})
            if "CloudCover" in dg:
                cc = dg["CloudCover"]
            else:
                add_attrs = u.get("AdditionalAttributes", [])
                for attr in add_attrs:
                    name = str(attr.get("Name", "")).lower()
                    if "cloud" in name:
                        vals = attr.get("Values", [])
                        if vals:
                            try:
                                cc = float(vals[0])
                            except Exception:
                                pass
                        break

        records.append(
            {
                "granule": g,
                "granule_id": granule_id,
                "start": pd.to_datetime(start_str) if start_str else pd.NaT,
                "cloud_cover": cc,
            }
        )

    df = pd.DataFrame.from_records(records)

    if cloud_max is not None and "cloud_cover" in df.columns:
        df = df[df["cloud_cover"].fillna(9999.0) <= cloud_max]

    if "start" in df.columns:
        df = df.sort_values("start").reset_index(drop=True)

    return results, df


In [3]:
from shapely.geometry import shape
from db_conn import get_folha_geom_geojson as folha
# alvo em UTC
target = pd.to_datetime("2008-06-15", utc=True)

granules, df = search_aster_l1t_004_cloudfree(
    bbox=shape(folha('SB21_ZA_II1_NE')).bounds,
    start="2000-01-01",
    end="2024-12-31",
    cloud_max=5.0,
)

if df.empty:
    print("Nenhum granule AST_L1T v004 encontrado para estes filtros.")
else:
    # garante que 'start' também está em UTC
    df["start"] = pd.to_datetime(df["start"], utc=True)

    df["delta_days"] = (df["start"] - target).abs().dt.days
    best = df.sort_values("delta_days").iloc[0]

    print(best[["granule_id", "start", "cloud_cover", "delta_days"]])

granule_id     AST_L1T_00406062008141311_20250622005853
start                  2008-06-06 14:13:11.722000+00:00
cloud_cover                                           0
delta_days                                            8
Name: 10, dtype: object


In [4]:
import earthaccess as ea
import rioxarray as rxr
import rasterio as rio
import matplotlib.pyplot as plt
from shapely.geometry import shape
from shapely.ops import transform as shp_transform
import pyproj
import pandas as pd

# geometria da folha em WGS84 (GeoJSON)
folha_id = 'SB21_ZA_II1_NE'
geom_folha = folha(folha_id)  # ou get_folha_geom_geojson(...)
geom_4326 = shape(geom_folha)

# alvo em UTC
target = pd.to_datetime("2008-06-15", utc=True)

# bbox em WGS84 para a busca
bbox = geom_4326.bounds

granules, df = search_aster_l1t_004_cloudfree(
    bbox=bbox,
    start="2000-01-01",
    end="2024-12-31",
    cloud_max=5.0,
)

if df.empty:
    print("Nenhum granule AST_L1T v004 encontrado para estes filtros.")
else:
    df["start"] = pd.to_datetime(df["start"], utc=True)
    df["delta_days"] = (df["start"] - target).abs().dt.days
    best = df.loc[df["delta_days"].idxmin()]
    print(best[["granule_id", "start", "cloud_cover", "delta_days"]])

    granule = best["granule"]

    download_dir = f"aster_{folha_id}"
    paths = ea.download([granule], download_dir)
    
    data_paths = [
        p for p in paths
        if p.suffix.lower() in (".hdf", ".h5", ".nc", ".tif", ".tiff")
    ]
    if not data_paths:
        raise RuntimeError(f"Nenhum arquivo de dado ASTER encontrado em: {paths}")
    
    data_path = data_paths[0]
    
    with rio.open(data_path) as src:
        da = rxr.open_rasterio(data_path, masked=True)
    
    # reprojeta geom da folha pro CRS do raster (se necessário)
    if da.rio.crs is not None and da.rio.crs.to_epsg() is not None and da.rio.crs.to_epsg() != 4326:
        transformer = pyproj.Transformer.from_crs("EPSG:4326", da.rio.crs, always_xy=True)
        geom_data = shp_transform(transformer.transform, geom_4326)
    else:
        geom_data = geom_4326
    
    minx, miny, maxx, maxy = geom_data.bounds
    
    da_clip = da.rio.clip_box(minx=minx, miny=miny, maxx=maxx, maxy=maxy)
    
    if "band" in da_clip.dims:
        da_clip.isel(band=0).plot.imshow(figsize=(6, 6))
    else:
        da_clip.plot.imshow(figsize=(6, 6))
    
    plt.title(best["granule_id"])
    plt.tight_layout()
    plt.axis('scaled')
    plt.show()


'NoneType' object has no attribute 'get': You must call earthaccess.login() before you can download data


granule_id     AST_L1T_00406062008141302_20250622005901
start                  2008-06-06 14:13:02.867000+00:00
cloud_cover                                           1
delta_days                                            8
Name: 9, dtype: object


RuntimeError: Nenhum arquivo de dado ASTER encontrado em: []

In [5]:
import earthaccess as ea
import rioxarray as rxr
import rasterio as rio
import matplotlib.pyplot as plt
from shapely.geometry import shape
from shapely.ops import transform as shp_transform
import pyproj
import pandas as pd


def process_aster_folha(
    folha_id: str,
    target_date: str = "2008-06-15",
    start_date: str = "2000-01-01",
    end_date: str = "2024-12-31",
    cloud_max: float = 5.0,
):
    print("\n================ ASTER L1T - PROCESSO =================")
    print(f"[INPUT] folha_id={folha_id}")
    print(f"[INPUT] target_date={target_date}, start={start_date}, end={end_date}, cloud_max={cloud_max}")

    # geometria da folha em WGS84 (GeoJSON)
    geom_folha = folha(folha_id)  # ou get_folha_geom_geojson(...)
    geom_4326 = shape(geom_folha)
    bbox = geom_4326.bounds
    print(f"[FOLHA] BBOX WGS84={bbox}")

    # alvo em UTC
    target = pd.to_datetime(target_date, utc=True)
    print(f"[TARGET] Data alvo (UTC)={target}")

    # busca ASTER
    print("[SEARCH] Buscando granules AST_L1T v004 no CMR/Earthdata...")
    granules, df = search_aster_l1t_004_cloudfree(
        bbox=bbox,
        start=start_date,
        end=end_date,
        cloud_max=cloud_max,
    )
    print(f"[SEARCH] granules retornados={len(granules)}, linhas em df={len(df)}")

    if df.empty:
        print("[SEARCH] Nenhum granule AST_L1T v004 encontrado para estes filtros.")
        return

    df["start"] = pd.to_datetime(df["start"], utc=True)
    df["delta_days"] = (df["start"] - target).abs().dt.days
    best_idx = df["delta_days"].idxmin()
    best = df.loc[best_idx]

    print("[SELECT] Melhor granule:")
    print(best[["granule_id", "start", "cloud_cover", "delta_days"]])

    granule = best["granule"]
    download_dir = f"aster_{folha_id}"
    print(f"[DOWNLOAD] Baixando dados para: {download_dir}")
    paths = ea.download([granule], download_dir)
    print(f"[DOWNLOAD] Arquivos baixados ({len(paths)}):")
    for p in paths:
        print(f"          - {p}")

    # escolhe arquivo de dado principal
    data_paths = [
        p for p in paths
        if p.suffix.lower() in (".hdf", ".h5", ".nc", ".tif", ".tiff")
    ]
    if not data_paths:
        raise RuntimeError(f"[ERROR] Nenhum arquivo de dado ASTER encontrado em: {paths}")

    data_path = data_paths[0]
    print(f"[DATA] Usando arquivo de dado: {data_path}")

    with rio.open(data_path) as src:
        print(f"[RASTER] CRS={src.crs}")
        print(f"[RASTER] width={src.width}, height={src.height}, bands={src.count}")

    da = rxr.open_rasterio(data_path, masked=True)
    print(f"[XR] dims={da.dims}, sizes={dict(da.sizes)}, crs={da.rio.crs}")

    # reprojeta geom da folha para o CRS do raster (se necessário)
    if da.rio.crs is not None and da.rio.crs.to_epsg() is not None and da.rio.crs.to_epsg() != 4326:
        print(f"[REPROJECT] Reprojetando folha de EPSG:4326 para {da.rio.crs}")
        transformer = pyproj.Transformer.from_crs("EPSG:4326", da.rio.crs, always_xy=True)
        geom_data = shp_transform(transformer.transform, geom_4326)
    else:
        print("[REPROJECT] CRS do raster é 4326 ou indefinido, usando geom_4326 diretamente")
        geom_data = geom_4326

    minx, miny, maxx, maxy = geom_data.bounds
    print(f"[CLIP] Bounds no CRS do raster: ({minx}, {miny}, {maxx}, {maxy})")

    da_clip = da.rio.clip_box(minx=minx, miny=miny, maxx=maxx, maxy=maxy)
    print(f"[CLIP] Shape após clip: {dict(da_clip.sizes)}")

    if "band" in da_clip.dims:
        print("[PLOT] Plotando banda 1 (index 0)...")
        da_clip.isel(band=0).plot.imshow(figsize=(6, 6))
    else:
        print("[PLOT] Plotando raster single-band...")
        da_clip.plot.imshow(figsize=(6, 6))

    plt.title(best["granule_id"])
    plt.tight_layout()
    plt.axis("scaled")
    plt.show()
    print("[DONE] Processo concluído.")


# chamada


In [6]:
process_aster_folha("SB21_ZA_II1_NE")


================ ASTER L1T - PROCESSO =================
[INPUT] folha_id=SB21_ZA_II1_NE
[INPUT] target_date=2008-06-15, start=2000-01-01, end=2024-12-31, cloud_max=5.0
[FOLHA] BBOX WGS84=(-56.375, -6.125, -56.25, -6.0)
[TARGET] Data alvo (UTC)=2008-06-15 00:00:00+00:00
[SEARCH] Buscando granules AST_L1T v004 no CMR/Earthdata...


'NoneType' object has no attribute 'get': You must call earthaccess.login() before you can download data


[SEARCH] granules retornados=35, linhas em df=35
[SELECT] Melhor granule:
granule_id     AST_L1T_00406062008141302_20250622005901
start                  2008-06-06 14:13:02.867000+00:00
cloud_cover                                           1
delta_days                                            8
Name: 9, dtype: object
[DOWNLOAD] Baixando dados para: aster_SB21_ZA_II1_NE
[DOWNLOAD] Arquivos baixados (0):


RuntimeError: [ERROR] Nenhum arquivo de dado ASTER encontrado em: []